# Full-Dataset DistilBERT Baselines

## Goal

Re-establish the main DistilBERT baselines on the complete GoEmotions
simplified dataset.

The earlier experiments used a 5,000-example training subset and a
500-example validation subset. This notebook uses the complete official
train, validation and test splits.

Experiments:
1. Plain DistilBERT multi-label baseline
2. Oversampling + class-weighted loss (OW)

Thresholds are selected on the validation split.
The test split is used only for final evaluation.

In [1]:
!pip install -q \
    transformers==4.40.2 \
    accelerate==0.27.2 \
    datasets \
    scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.0/321.0 kB 18.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.1 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [2]:
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "distilbert-base-uncased"

In [3]:
dataset = load_dataset(
    "google-research-datasets/go_emotions",
    "simplified",
)

dataset

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

simplified/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

In [4]:
label_names = dataset["train"].features["labels"].feature.names
num_labels = len(label_names)

tail_label_names = [
    "grief",
    "pride",
    "relief",
    "nervousness",
    "embarrassment",
    "remorse",
    "fear",
    "desire",
]

tail_label_ids = [
    label_names.index(label)
    for label in tail_label_names
]

print("Number of labels:", num_labels)
print("Tail labels:", tail_label_names)
print("Tail label IDs:", tail_label_ids)

Number of labels: 28
Tail labels: ['grief', 'pride', 'relief', 'nervousness', 'embarrassment', 'remorse', 'fear', 'desire']
Tail label IDs: [16, 21, 23, 19, 12, 24, 14, 8]


In [5]:
def encode_labels(example):
    multi_hot = np.zeros(num_labels, dtype=np.float32)

    for label_id in example["labels"]:
        multi_hot[label_id] = 1.0

    example["labels"] = multi_hot.tolist()
    return example


encoded_dataset = dataset.map(
    encode_labels,
    desc="Encoding labels",
)

encoded_dataset

Encoding labels:   0%|          | 0/43410 [00:00<?, ? examples/s]

Encoding labels:   0%|          | 0/5426 [00:00<?, ? examples/s]

Encoding labels:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )


tokenized_dataset = encoded_dataset.map(
    tokenize_batch,
    batched=True,
    desc="Tokenizing",
)

tokenized_dataset = tokenized_dataset.remove_columns(
    ["text", "id"]
)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing:   0%|          | 0/43410 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/5426 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [7]:
import torch
import torch.nn.functional as F
from transformers import Trainer


plain_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
)

plain_training_args = TrainingArguments(
    output_dir="./full_plain_distilbert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
    report_to="none",
)

import torch.nn.functional as F

class MultiLabelTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels").float()

        outputs = model(**inputs)
        logits = outputs.logits

        loss = F.binary_cross_entropy_with_logits(
            logits,
            labels,
        )

        return (loss, outputs) if return_outputs else loss

plain_trainer = MultiLabelTrainer(
    model=plain_model,
    args=plain_training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
test_batch = next(iter(plain_trainer.get_train_dataloader()))

print("Before compute_loss:")
print("labels dtype:", test_batch["labels"].dtype)
print("input_ids dtype:", test_batch["input_ids"].dtype)
print("labels shape:", test_batch["labels"].shape)

test_batch = {
    k: v.to(plain_model.device)
    for k, v in test_batch.items()
}

with torch.no_grad():
    test_loss = plain_trainer.compute_loss(
        plain_model,
        test_batch,
    )

print("Test loss:", test_loss.item())

Before compute_loss:
labels dtype: torch.int64
input_ids dtype: torch.int64
labels shape: torch.Size([16, 28])
Test loss: 0.6916261911392212


In [9]:
plain_train_result = plain_trainer.train()

Epoch,Training Loss,Validation Loss
1,0.093426,0.088851
2,0.081178,0.084007
3,0.072051,0.084055


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
plain_val_output = plain_trainer.predict(
    tokenized_dataset["validation"]
)

plain_val_logits = plain_val_output.predictions
plain_val_labels = plain_val_output.label_ids

plain_val_probs = torch.sigmoid(
    torch.tensor(plain_val_logits)
).numpy()

print(plain_val_probs.shape)
print(plain_val_labels.shape)

(5426, 28)
(5426, 28)


In [11]:
def evaluate_predictions(labels, probs, threshold):
    preds = (probs >= threshold).astype(int)

    micro_f1 = f1_score(
        labels,
        preds,
        average="micro",
        zero_division=0,
    )

    macro_f1 = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )

    tail_true = labels[:, tail_label_ids]
    tail_pred = preds[:, tail_label_ids]

    tail_precision = precision_score(
        tail_true,
        tail_pred,
        average="macro",
        zero_division=0,
    )

    tail_recall = recall_score(
        tail_true,
        tail_pred,
        average="macro",
        zero_division=0,
    )

    tail_f1 = f1_score(
        tail_true,
        tail_pred,
        average="macro",
        zero_division=0,
    )

    return {
        "threshold": threshold,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "tail_precision": tail_precision,
        "tail_recall": tail_recall,
        "tail_f1": tail_f1,
    }

In [12]:
thresholds = np.arange(0.05, 0.91, 0.05)

plain_threshold_results = []

for threshold in thresholds:
    result = evaluate_predictions(
        plain_val_labels,
        plain_val_probs,
        threshold,
    )
    plain_threshold_results.append(result)

plain_threshold_df = pd.DataFrame(
    plain_threshold_results
)

display(plain_threshold_df)

,threshold,micro_f1,macro_f1,tail_precision,tail_recall,tail_f1
0,0.05,0.454708,0.395594,0.324929,0.457257,0.322737
1,0.10,0.541522,0.458124,0.454976,0.402028,0.372198
2,0.15,0.579065,0.471170,0.375677,0.348224,0.346719
3,0.20,0.596347,0.468688,0.436737,0.309968,0.313643
4,0.25,0.605376,0.467087,0.336420,0.287621,0.300186
5,0.30,0.606395,0.461082,0.341186,0.269049,0.291171
6,0.35,0.604949,0.448233,0.361511,0.247135,0.277162
7,0.40,0.593358,0.429358,0.359188,0.228472,0.263212
8,0.45,0.578619,0.408874,0.360284,0.214150,0.251220
9,0.50,0.562902,0.386541,0.409897,0.190293,0.227635


In [13]:
plain_best_row = plain_threshold_df.loc[
    plain_threshold_df["macro_f1"].idxmax()
]

plain_best_threshold = float(
    plain_best_row["threshold"]
)

print("Best validation threshold by Macro-F1:")
display(plain_best_row)

Best validation threshold by Macro-F1:


,2
threshold,0.150000
micro_f1,0.579065
macro_f1,0.471170
tail_precision,0.375677
tail_recall,0.348224
tail_f1,0.346719


In [14]:
plain_test_output = plain_trainer.predict(
    tokenized_dataset["test"]
)

plain_test_logits = plain_test_output.predictions
plain_test_labels = plain_test_output.label_ids

plain_test_probs = torch.sigmoid(
    torch.tensor(plain_test_logits)
).numpy()

In [15]:
plain_test_metrics = evaluate_predictions(
    plain_test_labels,
    plain_test_probs,
    plain_best_threshold,
)

plain_test_metrics_df = pd.DataFrame(
    [plain_test_metrics]
)

display(plain_test_metrics_df)

,threshold,micro_f1,macro_f1,tail_precision,tail_recall,tail_f1
0,0.15,0.573941,0.462648,0.338006,0.337393,0.321204


# Experiment 51 – Full-dataset OW baseline

Train and evaluate the oversampling + class-weighted loss (OW) setup
on the complete GoEmotions dataset using the same validation/test protocol
as the plain full-dataset baseline.

In [16]:
# Original training split with multi-hot labels
train_dataset = encoded_dataset["train"]

# Mark examples that contain at least one tail label
def has_tail_label(example):
    return any(
        example["labels"][label_id] == 1.0
        for label_id in tail_label_ids
    )

tail_examples = train_dataset.filter(has_tail_label)

print("Original train size:", len(train_dataset))
print("Tail-containing examples:", len(tail_examples))

Filter:   0%|          | 0/43410 [00:00<?, ? examples/s]

Original train size: 43410
Tail-containing examples: 2533


In [17]:
from datasets import concatenate_datasets

ow_train_dataset = concatenate_datasets([
    train_dataset,
    tail_examples,
])

ow_train_dataset = ow_train_dataset.shuffle(seed=SEED)

print("OW train size:", len(ow_train_dataset))

OW train size: 45943


In [18]:
ow_label_matrix = np.array(
    ow_train_dataset["labels"],
    dtype=np.float32,
)

positive_counts = ow_label_matrix.sum(axis=0)
negative_counts = len(ow_label_matrix) - positive_counts

ow_pos_weight = negative_counts / positive_counts

ow_pos_weight_tensor = torch.tensor(
    ow_pos_weight,
    dtype=torch.float32,
)

pos_weight_df = pd.DataFrame({
    "label": label_names,
    "positive_count": positive_counts.astype(int),
    "pos_weight": ow_pos_weight,
})

display(pos_weight_df)

print(
    "pos_weight min / max / mean:",
    ow_pos_weight.min(),
    ow_pos_weight.max(),
    ow_pos_weight.mean(),
)

,label,positive_count,pos_weight
0,admiration,4217,9.894711
1,amusement,2364,18.434433
2,anger,1589,27.913153
3,annoyance,2519,17.238586
4,approval,2993,14.350150
5,caring,1132,39.585690
6,confusion,1388,32.100143
7,curiosity,2241,19.501116
8,desire,1282,34.836975
9,disappointment,1321,33.778954


pos_weight min / max / mean: 2.2118988 297.33118 54.540943


In [19]:
ow_tokenized_train = ow_train_dataset.map(
    tokenize_batch,
    batched=True,
    desc="Tokenizing OW train set",
)

ow_tokenized_train = ow_tokenized_train.remove_columns(
    ["text", "id"]
)

ow_tokenized_train.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

Tokenizing OW train set:   0%|          | 0/45943 [00:00<?, ? examples/s]

In [20]:
class OWTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels").float()

        outputs = model(**inputs)
        logits = outputs.logits

        pos_weight = self.pos_weight.to(logits.device)

        loss = F.binary_cross_entropy_with_logits(
            logits,
            labels,
            pos_weight=pos_weight,
        )

        return (loss, outputs) if return_outputs else loss

In [21]:
ow_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
ow_training_args = TrainingArguments(
    output_dir="./full_ow_distilbert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
    report_to="none",
)

In [24]:
ow_trainer = OWTrainer(
    model=ow_model,
    args=ow_training_args,
    train_dataset=ow_tokenized_train,
    eval_dataset=tokenized_dataset["validation"],
    pos_weight=ow_pos_weight_tensor,
)

In [25]:
test_batch = next(iter(ow_trainer.get_train_dataloader()))

test_batch = {
    k: v.to(ow_model.device)
    for k, v in test_batch.items()
}

with torch.no_grad():
    ow_test_loss = ow_trainer.compute_loss(
        ow_model,
        test_batch,
    )

print("OW sanity-check loss:", ow_test_loss.item())

OW sanity-check loss: 1.1370480060577393


In [26]:
ow_train_result = ow_trainer.train()

Epoch,Training Loss,Validation Loss
1,0.629487,0.590996
2,0.506744,0.570996
3,0.412403,0.595926


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [27]:
ow_val_output = ow_trainer.predict(
    tokenized_dataset["validation"]
)

ow_val_logits = ow_val_output.predictions
ow_val_labels = ow_val_output.label_ids

ow_val_probs = torch.sigmoid(
    torch.tensor(ow_val_logits)
).numpy()

In [28]:
ow_threshold_results = []

for threshold in thresholds:
    result = evaluate_predictions(
        ow_val_labels,
        ow_val_probs,
        threshold,
    )
    ow_threshold_results.append(result)

ow_threshold_df = pd.DataFrame(
    ow_threshold_results
)

display(ow_threshold_df)

,threshold,micro_f1,macro_f1,tail_precision,tail_recall,tail_f1
0,0.05,0.155363,0.131838,0.035624,0.928933,0.067550
1,0.10,0.209156,0.184598,0.057514,0.905395,0.105528
2,0.15,0.249387,0.223178,0.075148,0.884111,0.134322
3,0.20,0.283479,0.252738,0.089801,0.859472,0.157150
4,0.25,0.315134,0.280147,0.105583,0.843996,0.180737
5,0.30,0.342451,0.301800,0.118375,0.820852,0.198750
6,0.35,0.367121,0.322268,0.133395,0.819463,0.219833
7,0.40,0.388667,0.339684,0.146368,0.806693,0.236544
8,0.45,0.409996,0.357147,0.160352,0.802057,0.254700
9,0.50,0.429862,0.374506,0.178512,0.797656,0.277047


In [29]:
ow_best_row = ow_threshold_df.loc[
    ow_threshold_df["macro_f1"].idxmax()
]

ow_best_threshold = float(
    ow_best_row["threshold"]
)

print("Best OW validation threshold by Macro-F1:")
display(ow_best_row)

Best OW validation threshold by Macro-F1:


,17
threshold,0.900000
micro_f1,0.502541
macro_f1,0.489521
tail_precision,0.397215
tail_recall,0.625405
tail_f1,0.474739


In [30]:
ow_test_output = ow_trainer.predict(
    tokenized_dataset["test"]
)

ow_test_logits = ow_test_output.predictions
ow_test_labels = ow_test_output.label_ids

ow_test_probs = torch.sigmoid(
    torch.tensor(ow_test_logits)
).numpy()

ow_test_metrics = evaluate_predictions(
    ow_test_labels,
    ow_test_probs,
    ow_best_threshold,
)

ow_test_metrics_df = pd.DataFrame(
    [ow_test_metrics]
)

display(ow_test_metrics_df)

,threshold,micro_f1,macro_f1,tail_precision,tail_recall,tail_f1
0,0.9,0.491348,0.472149,0.330822,0.598614,0.412194


In [31]:
comparison_df = pd.DataFrame([
    {
        "model": "Plain",
        **plain_test_metrics,
    },
    {
        "model": "OW",
        **ow_test_metrics,
    },
])

display(comparison_df)

,model,threshold,micro_f1,macro_f1,tail_precision,tail_recall,tail_f1
0,Plain,0.15,0.573941,0.462648,0.338006,0.337393,0.321204
1,OW,0.90,0.491348,0.472149,0.330822,0.598614,0.412194
